# Algebra Error Classifier — Train + Eval (Colab)

Run **top to bottom** after every runtime disconnect.

| Set | Purpose |
|-----|---------|
| `test_holdout.jsonl` (~500, seed 2) | **Primary eval** — base vs fine-tuned |
| `val.jsonl` (seed 1) | Calibration only |
| `testset.jsonl` (12 hand-labeled) | Optional qualitative spot-check |

Runtime → **T4 GPU**. Save to Drive after training.


In [ ]:
import os

REPO = '/content/algebra_error_classifier'
GITHUB = 'https://github.com/gabriel-xiong/algebra_error_classifier.git'

if not os.environ.get('COLAB_GPU'):
    print('WARNING: No GPU. Use Runtime -> Change runtime type -> T4 GPU.')

!rm -rf {REPO}
!git clone {GITHUB} {REPO}
os.chdir(REPO)
print('cwd:', os.getcwd())
!ls data/


In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install trl datasets peft matplotlib pyyaml


In [ ]:
# Verify repo scripts (catch stale -Instruct model id)
import os
from pathlib import Path

MODEL = 'Qwen/Qwen3-1.7B'
os.environ['MODEL'] = MODEL

bad = []
for path in Path('scripts').glob('*.py'):
    text = path.read_text(encoding='utf-8')
    if '1.7B-Instruct' in text:
        bad.append(str(path))
if bad:
    raise SystemExit('Stale scripts still reference -Instruct: ' + ', '.join(bad))

!grep "add_argument\(\\"--model\\"" scripts/train_sft.py
print('MODEL =', MODEL)


In [ ]:
# Data ships in the repo (disjoint train/val/test_holdout from make_splits.py).
# Regenerate only if you want a fresh pool or different sizes.
from pathlib import Path

REGENERATE = False  # set True to rebuild splits (thousands of records)

if REGENERATE or not Path('data/train.jsonl').exists():
    !python scripts/make_splits.py --train-n 6000 --val-n 800 --test-n 1000 --seed 0 --out-dir data
    !python scripts/prepare_sft.py --data data/train.jsonl --out data/train_sft.jsonl
    !python scripts/prepare_sft.py --data data/val.jsonl --out data/val_sft.jsonl

!wc -l data/train.jsonl data/val.jsonl data/test_holdout.jsonl data/train_sft.jsonl data/val_sft.jsonl


## Optional — restore from Drive


In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# DEST = '/content/drive/MyDrive/algebra_error_classifier_outputs'
# if __import__('os').path.exists(DEST):
#     shutil.copytree(DEST, 'outputs', dirs_exist_ok=True)
#     print('Restored from Drive')


## 1. Baseline eval on held-out test (~1,000 examples)


In [ ]:
!python scripts/run_baseline.py --model Qwen/Qwen3-1.7B --data data/test_holdout.jsonl --runs 1 --out-dir outputs/eval/base_holdout --save-predictions outputs/eval/base_holdout_predictions.jsonl


## 2. QLoRA fine-tuning (~30–45 min)


In [ ]:
!python scripts/train_sft.py --model Qwen/Qwen3-1.7B --train data/train_sft.jsonl --val data/val_sft.jsonl --out outputs/lora --epochs 2 --batch-size 4 --grad-accum 4


## Save to Drive (run right after training)


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/algebra_error_classifier_outputs'
shutil.copytree('outputs', DEST, dirs_exist_ok=True)
print('Saved to', DEST)


## 3. Calibration (val only — seed 1)


In [ ]:
!python scripts/calibrate.py --model Qwen/Qwen3-1.7B --adapter outputs/lora --data data/val.jsonl --out outputs/calibration.json --max-examples 100


## 4. Tuned eval on held-out test (~1,000 examples)


In [ ]:
!python scripts/run_baseline.py --model Qwen/Qwen3-1.7B --adapter outputs/lora --data data/test_holdout.jsonl --calibration outputs/calibration.json --runs 1 --out-dir outputs/eval/tuned_holdout --save-predictions outputs/eval/tuned_holdout_predictions.jsonl


## Optional — hand-labeled spot-check (12 examples)


In [ ]:
# !python scripts/run_baseline.py --model Qwen/Qwen3-1.7B --adapter outputs/lora --data data/testset.jsonl --calibration outputs/calibration.json --runs 5 --out-dir outputs/eval/tuned_litmus


## 5. Save everything to Drive again


In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
DEST = '/content/drive/MyDrive/algebra_error_classifier_outputs'
shutil.copytree('outputs', DEST, dirs_exist_ok=True)
print('Updated', DEST)


## Optional — hard-negative mining (concentrate on cases the base model fails)

Score the training pool with the base model, keep the wrong / overconfident-on-abstain
cases, upweight them, and retrain on the augmented set. This is the defensible version
of "train on what LLMs are bad at."


In [ ]:
# !python scripts/mine_hard_negatives.py --model Qwen/Qwen3-1.7B \
#   --pool data/train.jsonl --out data/hard_negatives.jsonl \
#   --augment-train data/train.jsonl --weight 3 --out-augmented data/train_hardaug.jsonl
# !python scripts/prepare_sft.py --data data/train_hardaug.jsonl --out data/train_hardaug_sft.jsonl
# !python scripts/train_sft.py --model Qwen/Qwen3-1.7B \
#   --train data/train_hardaug_sft.jsonl --val data/val_sft.jsonl --out outputs/lora_hardaug --epochs 2


## Optional — real-data eval (Eedi / DataShop mapped to taxonomy)

Upload a real dump to `data/real_eval.jsonl` (see `docs/real_data.md`), then compare
base vs tuned on real student errors. `--normalize-real` maps varied field names +
free-text misconceptions onto the 7 labels.


In [ ]:
# !python scripts/run_baseline.py --model Qwen/Qwen3-1.7B \
#   --data data/real_eval.jsonl --normalize-real --score-labels \
#   --calibration outputs/calibration.json --out-dir outputs/eval/real_base
# !python scripts/run_baseline.py --model Qwen/Qwen3-1.7B --adapter outputs/lora \
#   --data data/real_eval.jsonl --normalize-real --score-labels \
#   --calibration outputs/calibration.json --out-dir outputs/eval/real_tuned
